# Let's go PRO!

Advanced RAG Techniques!

Let's start by digging into ingest:

1. No LangChain! Just native for maximum flexibility
2. Let's use an LLM to divide up chunks in a sensible way
3. Let's use the best chunk size and encoder from yesterday
4. Let's also have the LLM rewrite chunks in a way that's most useful ("document pre-processing")


In [2]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import (
    PersistentClient,
)  # instead of using LangChain, we are using chroma's open source packages directly
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go


load_dotenv(override=True)

MODEL = "gpt-4.1-nano"

DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

openai = OpenAI()

In [ ]:
# Inspired by LangChain's Document - let's have something similar (see week 5 day 2)


class Result(BaseModel):
    page_content: str
    metadata: dict

In [ ]:
# A class to perfectly represent a chunk


class Chunk(BaseModel):
    headline: str = Field(
        description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query"
    )
    summary: str = Field(
        description="A few sentences summarizing the content of this chunk to answer common questions"
    )  # something new that is added to our chunks
    original_text: str = Field(
        description="The original text of this chunk from the provided document, exactly as is, not changed in any way"
    )

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(
            page_content=self.headline
            + "\n\n"
            + self.summary
            + "\n\n"
            + self.original_text,
            metadata=metadata,
        )


class Chunks(BaseModel):
    chunks: list[Chunk]

## Three steps:

1. Fetch documents from the knowledge base, like LangChain did
2. Call an LLM to turn documents into Chunks
3. Store the Chunks in Chroma

That's it!

### Let's start with Step 1


In [ ]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader (see week 5 day 2)"""

    documents = []

    # look into the directory for our knowledge base, and read the files if it is markdown and then you add it into the list
    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append(
                    {"type": doc_type, "source": file.as_posix(), "text": f.read()}
                )

    print(f"Loaded {len(documents)} documents")
    return documents

In [6]:
documents = fetch_documents()

Loaded 76 documents


### Donezo! On to Step 2 - make the chunks


In [7]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [8]:
print(make_prompt(documents[0]))


You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: products
The document has been retrieved from: knowledge-base/products/Rellm.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into 8 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

# Product Summary

# Rellm: AI-Powered Enterprise 

In [9]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [10]:
make_messages(documents[0])

[{'role': 'user',
  'content': "\nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the shared drive of a company called Insurellm.\nThe document is of type: products\nThe document has been retrieved from: knowledge-base/products/Rellm.md\n\nA chatbot will use these chunks to answer questions about the company.\nYou should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.\nThis document should probably be split into 8 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a headline, a summary, and the original text of the chunk.\nTogether your chunks should represent the entire document with overlap.\n\nHere is the document:\n\n#

In [ ]:
def process_document(document):
    messages = make_messages(document)
    response = completion(
        model=MODEL, messages=messages, response_format=Chunks
    )  # this litellm is very similar to openAI's openai.chat.completions.create
    # when doing structured outputs, make sure to provide a 'response_format' argument
    # Chunks is a list of Chunk that we defined earlier
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [12]:
process_document(documents[0])

[Result(page_content='Product Overview of Rellm\n\nRellm is an enterprise reinsurance platform developed by Insurellm that utilizes AI to improve risk management, decision-making, and operational efficiency in reinsurance companies.\n\n# Product Summary\n\n# Rellm: AI-Powered Enterprise Reinsurance Solution\n\n## Summary\n\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.', metadata={'source': 'knowledge-base/products/Rellm.md', 'type': 'products'}),
 Result(page_content="Key Features of Rellm - Part 1\n\nRellm offers AI-driv

In [13]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        # we usually expect 'for doc in documents', here tqdm() is a data sci utility that will add a little status bar as we iterate over the documents (we always see them when using huggingface)
        chunks.extend(process_document(doc))
    return chunks

In [14]:
chunks = create_chunks(
    documents
)  # it's going to call gpt-4.1-nano 76 times, once for each document.

100%|██████████| 76/76 [10:20<00:00,  8.16s/it]


In [15]:
print(len(chunks))

373


### Well that was easy! If a bit slow.

In the python module version, I sneakily use the multi-processing Pool to run this in parallel,
but if you get a Rate Limit Error you can turn this off in the code.

### Finally, Step 3 - save the embeddings


In [16]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(
        model=embedding_model, input=texts
    ).data  # the database of embeddings is separate from the action of embedding the docuements
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [17]:
create_embeddings(chunks)

Vectorstore created with 373 documents


# Nothing more to do here... right?

Wait! Didja think I'd forget??


In [18]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=["embeddings", "documents", "metadatas"])
vectors = np.array(result["embeddings"])
documents = result["documents"]
metadatas = result["metadatas"]
doc_types = [metadata["type"] for metadata in metadatas]
colors = [
    ["blue", "green", "red", "orange"][
        ["products", "employees", "contracts", "company"].index(t)
    ]
    for t in doc_types
]

In [ ]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(
    data=[
        go.Scatter(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            mode="markers",
            marker=dict(size=5, color=colors, opacity=0.8),
            text=[
                f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)
            ],
            hoverinfo="text",
        )
    ]
)

fig.update_layout(
    title="2D Chroma Vector Store Visualization",
    scene=dict(xaxis_title="x", yaxis_title="y"),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40),
)

fig.show()

In [ ]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            z=reduced_vectors[:, 2],
            mode="markers",
            marker=dict(size=5, color=colors, opacity=0.8),
            text=[
                f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)
            ],
            hoverinfo="text",
        )
    ]
)

fig.update_layout(
    title="3D Chroma Vector Store Visualization",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40),
)

fig.show()

## And now - let's build an Advanced RAG!

We will use these techniques:

1. Reranking - reorder the rank results
2. Query re-writing


In [21]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [22]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(
        model=MODEL, messages=messages, response_format=RankOrder
    )  # use of structued outputs again
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

In [30]:
RETRIEVAL_K = 10


# this is the alternative of not using langchain, but just doing the retrieval ourselves.
# langchain is like this: retriever.invoke("Who is Avery?")
def fetch_context_unranked(question):
    query = (
        openai.embeddings.create(model=embedding_model, input=[question])
        .data[0]
        .embedding
    )  # turn ths question into an embedding

    results = collection.query(
        query_embeddings=[query], n_results=RETRIEVAL_K
    )  # query the collection given this query and say that we want k results
    chunks = []
    # print(zip(results["documents"][0], results["metadatas"][0]))
    for result in zip(results["documents"][0], results["metadatas"][0]):
        # so zip(list1, list2) basically pairs the things together, so item 1 in list 1 is paired with item 1 in list 2, result is just a tuple of (item 1 in list 1, item 1 in list 2)
        # you can unwrap it in to 'document, meta' instead of just 'result'
        # print(result)
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [25]:
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)

("Career Progression Timeline (Part 2)\n\nDetails of Maxine Thompson's promotion to Senior Data Engineer and her achievements from 2021 onwards.\n\n## Insurellm Career Progression\n- **January 2021 - Present**: **Senior Data Engineer**  \n  * Maxine was promoted to Senior Data Engineer after successfully leading a pivotal project that improved data retrieval times by 30%. She now mentors junior engineers and is involved in strategic data initiatives, solidifying her position as a valued asset at Insurellm. She was recognized as Insurellm Innovator of the year in 2023, receiving the prestigious IIOTY 2023 award.", {'type': 'employees', 'source': 'knowledge-base/employees/Maxine Thompson.md'})
('Performance History Overview\n\nThis chunk presents Oliver\'s annual performance evaluations from 2018 to 2023, showing fluctuations from steady improvement to challenges, and showcasing a strong comeback in 2022.\n\n## Annual Performance History\n- **2018**: **3/5** - Adaptable team player but s

In [26]:
for chunk in chunks:
    print(chunk.page_content[:15] + "...")

Career Progress...
Performance His...
Career Progress...
Annual Performa...
Annual Performa...
Annual Performa...
Performance His...
Annual Performa...
Career Progress...
Performance Rev...


In [27]:
reranked = rerank(question, chunks)

[1, 10, 2, 3, 4, 5, 6, 7, 8, 9]


In [28]:
for chunk in reranked:
    print(chunk.page_content[:15] + "...")

Career Progress...
Performance Rev...
Performance His...
Career Progress...
Annual Performa...
Annual Performa...
Annual Performa...
Performance His...
Annual Performa...
Career Progress...


In [ ]:
question = "Who went to Manchester University?"  # the normal RAG model in days 3 and 4 cannot answer this question because it is buried deep inside 1 of the HR records, but after we did semantic chunking (using LLM to break to text to chunks first) it performs better
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)  # if it finds the document, it will print the index

In [38]:
reranked = rerank(question, chunks)
# this output means that the previously 4th ranked document is now shifted to the top.

[4, 9, 14, 12, 20, 1, 2, 3, 5, 6, 7, 8, 10, 11, 13, 15, 16, 17, 18, 19]


In [39]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)

In [40]:
reranked[0].page_content

"Insurellm Career Progression (Part 1)\n\nThis part covers Priya Sharma's career at Insurellm since March 2018, highlighting her roles and achievements over the years.\n\n## Insurellm Career Progression\n- **March 2018 - Present:** Senior Data Scientist\n  - Leads machine learning initiatives for risk prediction models\n  - Built recommendation engine for Marketllm increasing conversion by 28%\n  - Mentors team of 3 junior data scientists\n  - Published 2 research papers on insurance ML applications"

In [36]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [ ]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

# this line is added to improve our prompt: Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.

In [42]:
# In the context, include the source of the chunk


def make_rag_messages(question, history, chunks):
    context = "\n\n".join(
        f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}"
        for chunk in chunks
    )
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": question}]
    )

In [ ]:
def rewrite_query(question, history=[]):
    # this function optionally takes in history
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(
        model=MODEL, messages=[{"role": "system", "content": message}]
    )
    return response.choices[0].message.content


# rewriting the query sometimes add words like 'insurellm' to the question, diluting the question posed by the user, and sometimes the relevant chunks does not get surfaced up, like Jessica Liu's HR records stating she attended Manchester University
# Don't mention the company name unless it's a general question about the company. -> that's why we added this line, but sometimes they will still add it.

In [44]:
rewrite_query("Who won the IIOTY award?", [])

'Who was the recipient of the IIOTY award?'

In [45]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

In [ ]:
answer_question("Who won the IIOTY award?", [])
# a good thing about re-ranking is you can also chop off the last 10 or so chunks because they are irrelevant, we don't need to pass them in future calls

Who received the IIOTY award?
[1, 8, 13, 11, 16, 17, 20, 3, 4, 6, 2, 5, 7, 9, 10, 12, 14, 15, 18, 19]


('Maxine Thompson was recognized as the Insurellm Innovator of the Year (IIOTY) in 2023.',
 [Result(page_content="Career Progression Timeline (Part 2)\n\nDetails of Maxine Thompson's promotion to Senior Data Engineer and her achievements from 2021 onwards.\n\n## Insurellm Career Progression\n- **January 2021 - Present**: **Senior Data Engineer**  \n  * Maxine was promoted to Senior Data Engineer after successfully leading a pivotal project that improved data retrieval times by 30%. She now mentors junior engineers and is involved in strategic data initiatives, solidifying her position as a valued asset at Insurellm. She was recognized as Insurellm Innovator of the year in 2023, receiving the prestigious IIOTY 2023 award.", metadata={'type': 'employees', 'source': 'knowledge-base/employees/Maxine Thompson.md'}),
  Result(page_content="Performance Review Highlights (2021-2023)\n\nSummary of Maxine Thompson's outstanding and exceeding expectation reviews in recent years.\n\n## Annual Perf

In [ ]:
answer_question(
    "Who went to Manchester University?", []
)  # this doesn't always work - due to query rewriting

Which individual attended Manchester University?
[16, 11, 1, 15, 13, 7, 8, 17, 14, 18, 19, 3, 2, 4, 5, 6, 9, 10, 12, 20]


("I don't have any information indicating that someone from Insurellm attended Manchester University.",
 [Result(page_content="Career Progression at Insurellm\n\nDetails Rachel Martinez's career development at Insurellm, highlighting her roles from Associate Product Manager to Product Manager, along with key responsibilities and achievements.\n\n## Insurellm Career Progression\n- **March 2019 - Present:** Product Manager\n  - Leads product strategy for Carllm, the auto insurance portal\n  - Successfully launched three major feature releases that increased user engagement by 45%\n  - Manages cross-functional teams including engineering, design, and sales\n\n- **January 2017 - February 2019:** Associate Product Manager\n  - Supported product development for Marketllm marketplace\n  - Conducted user research and competitive analysis\n  - Collaborated with engineering teams on feature prioritization\n\n- **June 2015 - December 2016:** Business Analyst at TechInsure Corp\n  - Analyzed marke